## File Upload Location

**Upload your CSV/JSON files from Windows to this location:**

```
/Volumes/main/crypto_bronze/raw_data/
```

### How to upload:
1. In Databricks UI, go to: **Catalog** → **main** → **crypto_bronze** → **raw_data** (volume)
2. Click **Upload** button
3. Select your `crypto_data_*.csv` or `crypto_data_*.json` file
4. Run this notebook to process the uploaded file

### Alternative method:
Drag and drop files directly into the Databricks workspace file browser at:
```
/Volumes/main/crypto_bronze/raw_data/
```

In [0]:
# WORKAROUND: Network connectivity issue - generating sample data
# The Serverless compute cannot reach api.coingecko.com due to DNS resolution
# This cell creates realistic sample data for development/testing

from datetime import datetime, timezone
import json

# Sample CoinGecko market data (realistic structure)
sample_data = [
    {
        "id": "bitcoin",
        "symbol": "btc",
        "name": "Bitcoin",
        "image": "https://assets.coingecko.com/coins/images/1/large/bitcoin.png",
        "current_price": 65432.50,
        "market_cap": 1280456789012,
        "market_cap_rank": 1,
        "fully_diluted_valuation": 1374567890123,
        "total_volume": 28456789012,
        "high_24h": 66123.45,
        "low_24h": 64890.23,
        "price_change_24h": 542.27,
        "price_change_percentage_24h": 0.84,
        "market_cap_change_24h": 10567890123,
        "market_cap_change_percentage_24h": 0.83,
        "circulating_supply": 19567890.0,
        "total_supply": 21000000.0,
        "max_supply": 21000000.0,
        "ath": 69045.0,
        "ath_change_percentage": -5.24,
        "ath_date": "2021-11-10T14:24:11.849Z",
        "atl": 67.81,
        "atl_change_percentage": 96408.52,
        "atl_date": "2013-07-06T00:00:00.000Z",
        "last_updated": "2026-08-26T12:35:00.000Z"
    },
    {
        "id": "ethereum",
        "symbol": "eth",
        "name": "Ethereum",
        "image": "https://assets.coingecko.com/coins/images/279/large/ethereum.png",
        "current_price": 3245.67,
        "market_cap": 389876543210,
        "market_cap_rank": 2,
        "fully_diluted_valuation": None,
        "total_volume": 15678901234,
        "high_24h": 3289.45,
        "low_24h": 3198.12,
        "price_change_24h": 47.55,
        "price_change_percentage_24h": 1.49,
        "market_cap_change_24h": 5789012345,
        "market_cap_change_percentage_24h": 1.51,
        "circulating_supply": 120123456.0,
        "total_supply": 120123456.0,
        "max_supply": None,
        "ath": 4878.26,
        "ath_change_percentage": -33.47,
        "ath_date": "2021-11-10T14:24:19.604Z",
        "atl": 0.432979,
        "atl_change_percentage": 749087.23,
        "atl_date": "2015-10-20T00:00:00.000Z",
        "last_updated": "2026-08-26T12:35:00.000Z"
    },
    {
        "id": "solana",
        "symbol": "sol",
        "name": "Solana",
        "image": "https://assets.coingecko.com/coins/images/4128/large/solana.png",
        "current_price": 142.35,
        "market_cap": 65432109876,
        "market_cap_rank": 5,
        "fully_diluted_valuation": 82345678901,
        "total_volume": 2345678901,
        "high_24h": 145.67,
        "low_24h": 140.23,
        "price_change_24h": 2.12,
        "price_change_percentage_24h": 1.51,
        "market_cap_change_24h": 987654321,
        "market_cap_change_percentage_24h": 1.53,
        "circulating_supply": 459876543.0,
        "total_supply": 578901234.0,
        "max_supply": None,
        "ath": 259.96,
        "ath_change_percentage": -45.24,
        "ath_date": "2021-11-06T21:54:35.825Z",
        "atl": 0.500801,
        "atl_change_percentage": 28315.67,
        "atl_date": "2020-05-11T19:35:23.449Z",
        "last_updated": "2026-08-26T12:35:00.000Z"
    }
]

data = sample_data
print(f"✓ Generated {len(data)} sample records (WORKAROUND for network issue)")
print(f"\nNote: To use live data, run this notebook on compute with external API access")

In [0]:
import json
import os
from datetime import datetime, timezone

# ==============================================================================
# FILE-BASED INGESTION (instead of direct API call)
# Upload your CSV/JSON file to: /Volumes/main/crypto_bronze/raw_data/
# ==============================================================================

# Define the upload location
upload_path = "/Volumes/main/crypto_bronze/raw_data/"

# Create volume if it doesn't exist
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS main.crypto_bronze")
    spark.sql("CREATE VOLUME IF NOT EXISTS main.crypto_bronze.raw_data")
    print(f"✓ Volume ready: {upload_path}")
except Exception as e:
    print(f"Volume may already exist: {e}")

# List available files in the upload location
try:
    files = dbutils.fs.ls(upload_path)
    print(f"\n📁 Files in {upload_path}:")
    
    if not files:
        print("   (No files found - please upload your data file)")
        print("\n⚠️ UPLOAD REQUIRED:")
        print("   1. Run the local_windows_ingestion.py script on your Windows PC")
        print("   2. Upload the generated CSV/JSON file to Databricks:")
        print(f"      {upload_path}")
        print("   3. Re-run this cell to process the file")
        raise FileNotFoundError("No data files found in upload location")
    
    for file in files:
        print(f"   - {file.name} ({file.size} bytes)")
    
    # Find the most recent JSON or CSV file
    json_files = [f for f in files if f.name.endswith('.json')]
    csv_files = [f for f in files if f.name.endswith('.csv')]
    
    data_file = None
    file_type = None
    
    if json_files:
        # Use the most recent JSON file
        data_file = max(json_files, key=lambda f: f.modificationTime)
        file_type = "json"
    elif csv_files:
        # Use the most recent CSV file
        data_file = max(csv_files, key=lambda f: f.modificationTime)
        file_type = "csv"
    else:
        raise FileNotFoundError("No JSON or CSV files found. Please upload data files.")
    
    file_path = data_file.path
    print(f"\n✓ Processing file: {data_file.name} ({file_type.upper()})")
    print(f"  Size: {data_file.size} bytes")
    print(f"  Modified: {datetime.fromtimestamp(data_file.modificationTime/1000)}")
    
    # Read the file based on type
    if file_type == "json":
        # Read JSON file
        file_content = dbutils.fs.head(file_path, 10000000)  # Read up to 10MB
        data = json.loads(file_content)
        print(f"✓ Successfully loaded {len(data)} records from JSON file")
    
    elif file_type == "csv":
        # Read CSV file using Spark
        df_csv = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(file_path)
        
        # Convert to list of dictionaries
        data = [row.asDict() for row in df_csv.collect()]
        print(f"✓ Successfully loaded {len(data)} records from CSV file")
    
    print(f"\n✓ Data ingestion complete from uploaded file!")
    
except Exception as e:
    print(f"\n⚠️ Error reading file: {e}")
    print("\n📋 INSTRUCTIONS:")
    print("   1. Download and run 'local_windows_ingestion.py' on your Windows machine")
    print("   2. Upload the generated file (crypto_data_*.json or crypto_data_*.csv)")
    print(f"   3. Upload location: {upload_path}")
    print("   4. Re-run this cell")
    raise

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.functions import from_json, col
import json

# Convert to DataFrame - CoinGecko returns list of dictionaries
df_raw = spark.createDataFrame(data)

print(f"✓ Created DataFrame with {df_raw.count()} rows")
print(f"\nSchema:")
df_raw.printSchema()

display(df_raw)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    sha2,
    concat_ws,
    to_json,
    struct
)

# Add bronze layer metadata columns
df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("CoinGecko"))
    .withColumn("source_endpoint", lit("/api/v3/coins/markets"))
    .withColumn("ingestion_date", current_timestamp().cast("date"))
    .withColumn("record_hash", sha2(to_json(struct("*")), 256))
)

print(f"✓ Added bronze layer metadata")
print(f"Total columns: {len(df_bronze.columns)}")
display(df_bronze)

In [0]:
# Create schema if it doesn't exist (using main catalog)
spark.sql("CREATE SCHEMA IF NOT EXISTS main.crypto_bronze")

# Define bronze table path (fully qualified: catalog.schema.table)
bronze_table = "main.crypto_bronze.coingecko_market_data"

# Write to Delta table with append mode
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(bronze_table)

record_count = df_bronze.count()
print(f"✓ Successfully wrote {record_count} records to {bronze_table}")
print(f"Timestamp: {datetime.now(timezone.utc).isoformat()}")

In [0]:
# Verify the data in bronze table
bronze_table = "main.crypto_bronze.coingecko_market_data"
df_verify = spark.table(bronze_table)

print(f"✓ Bronze table verification:")
print(f"Total records: {df_verify.count()}")
print(f"\nLatest ingestion:")
df_verify.select("id", "name", "symbol", "current_price", "market_cap", "ingestion_timestamp") \
    .orderBy(col("ingestion_timestamp").desc()) \
    .show(10, truncate=False)